# W3 Challenge: Customer Sentiment at Scale (50K)

**Author:** Alberto Diaz Durana  
**Date:** March 2026  
**Course:** AI in Data Science

## Objective

Repeat the W3 sentiment analysis on the full Customer Support on Twitter dataset (~2.8M tweets) with a 50K sample to test how the tools and findings scale. This is the larger-sample version of the 10K challenge notebook.

## Structure

1. Setup and Data Loading
2. Gemini Exploration (full dataset statistics)
3. AutoViz Visualization (sampled)
4. Sentiment Analysis (GPU-accelerated, 50K sample)
5. Company-Level Analysis
6. Scale Comparison and Reflections

In [1]:
# Cell 1 - Installs and Imports
import subprocess, sys

packages = [
    'google-genai', 'autoviz', 'transformers', 'torch',
    'pandas', 'matplotlib', 'seaborn'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, re, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from transformers import pipeline
import torch
import warnings

print(f"Python: {sys.version}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("All imports successful.")

Python: 3.10.12 (main, Jan 26 2026, 14:55:28) [GCC 11.4.0]
GPU available: True
GPU: Quadro T1000 with Max-Q Design
All imports successful.


In [2]:
# Cell 2 - Data Loading and Initial Inspection
import os

# Local paths (challenge uses full dataset)
local_paths = ['data/twcs/twcs.csv', 'data/twcs.csv']

try:
    import google.colab
    data_dir = '/content/data'
    local_paths = [os.path.join(data_dir, 'twcs.csv')]
except ImportError:
    data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')

df = None
for path in local_paths:
    full_path = path if os.path.isabs(path) else os.path.join(os.path.dirname(os.getcwd()), path)
    if os.path.exists(full_path):
        print(f"Loading from: {full_path}")
        df = pd.read_csv(full_path)
        break

assert df is not None, "Dataset not found. Download: kaggle datasets download -d thoughtvector/customer-support-on-twitter"

print(f"\nShape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nInbound: {df['inbound'].sum():,} ({df['inbound'].mean():.1%})")
print(f"Outbound: {(~df['inbound']).sum():,} ({(~df['inbound']).mean():.1%})")
print(f"\nTop 10 companies (outbound):")
print(df[df['inbound'] == False]['author_id'].value_counts().head(10))
print(f"\nSample rows:")
print(df.head(3).to_string())

Loading from: /home/berto/_projects/AI-in-Data-Science/data/twcs/twcs.csv

Shape: 2,811,774 rows, 7 columns

Column types:
tweet_id                     int64
author_id                   object
inbound                       bool
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object

Missing values:
tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64

Inbound: 1,537,843 (54.7%)
Outbound: 1,273,931 (45.3%)

Top 10 companies (outbound):
author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airwa

In [3]:
# Cell 3 - Gemini Exploration (Full Dataset)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
except (ImportError, Exception):
    env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.strip().startswith('GOOGLE_API_KEY='):
                    api_key = line.strip().split('=', 1)[1]
if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')

assert api_key, "GOOGLE_API_KEY not found."
client = genai.Client(api_key=api_key)

# Build summary with full dataset stats
inbound = df[df['inbound'] == True]
outbound = df[df['inbound'] == False]
top_companies = outbound['author_id'].value_counts().head(15)

dataset_summary = f"""Dataset: Customer Support on Twitter (FULL)
Shape: {df.shape[0]:,} rows, {df.shape[1]} columns
Columns: {list(df.columns)}

Inbound (customer) tweets: {len(inbound):,} ({len(inbound)/len(df):.1%})
Outbound (company) tweets: {len(outbound):,} ({len(outbound)/len(df):.1%})

Missing values:
{df.isnull().sum().to_string()}

Top 15 companies by response volume:
{top_companies.to_string()}

Sample inbound tweets:
{inbound['text'].sample(5, random_state=42).to_string()}

Sample outbound tweets:
{outbound['text'].sample(5, random_state=42).to_string()}
"""

prompt = f"""You are a data analyst. Given the following dataset summary from the full Customer Support on Twitter corpus (~2.8M tweets), please:
1. Describe the dataset structure and scale
2. Identify patterns in company representation and tweet distribution
3. Suggest analytical directions for sentiment analysis at scale

{dataset_summary}"""

print("Sending to Gemini...\n")
response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
print("=== Gemini's Analysis ===\n")
print(response.text)

Sending to Gemini...

=== Gemini's Analysis ===

As a data analyst, here's my assessment of the provided Customer Support on Twitter dataset summary:

---

### 1. Dataset Structure and Scale

**Scale:**
*   **Total Records:** The dataset is substantial, comprising **2,811,774 tweets**. This makes it a large-scale corpus for analyzing customer support interactions.
*   **Dimensionality:** It has **7 distinct columns**, providing core information about each tweet and its conversational context.

**Structure (Columns and their implications):**

*   **`tweet_id` (Unique Identifier):** Essential for uniquely identifying each tweet. No missing values, which is excellent for data integrity.
*   **`author_id` (User Identifier):** Identifies the author of the tweet. Crucial for grouping tweets by user (customer or company) and analyzing company-specific interactions. No missing values.
*   **`inbound` (Boolean/Flag):** A binary indicator (`True`/`False` or similar) signifying whether the tweet 

In [4]:
# Cell 4 - AutoViz Visualization (Sampled)
warnings_warn = warnings.warn
warnings.warn = lambda *a, **kw: None

import IPython.core.pylabtools as _pylabtools
_pylabtools.backend2gui = {}

# Sample and add derived features
sample_viz = df.sample(10000, random_state=42).copy()
sample_viz['text_length'] = sample_viz['text'].str.len()
sample_viz['word_count'] = sample_viz['text'].str.split().str.len()

viz_df = sample_viz[['author_id', 'inbound', 'text_length', 'word_count']].copy()
viz_df['inbound'] = viz_df['inbound'].astype(str)

from autoviz import AutoViz_Class
AV = AutoViz_Class()
report = AV.AutoViz(
    filename='', dfte=viz_df, depVar='', verbose=1,
    lowess=False, chart_format='svg',
    max_rows_analyzed=10000, max_cols_analyzed=10
)

warnings.warn = warnings_warn
print(f"\nAutoViz completed on 10K sample.")

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)
Shape of your Data Set loaded: (10000, 4)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  0
    Number of Integer-Categorical Columns =  2
    Number of String-Categorical Columns =  0
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  1
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  1
    Number of

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
author_id,object,0.000000,58,,,No issue
inbound,object,0.000000,0,,,No issue
text_length,int64,0.000000,3,2.000000,392.000000,Column has 405 outliers greater than upper bound (234.50) or lower than lower bound(-17.50). Cap them or remove them.
word_count,int64,0.000000,0,1.000000,60.000000,"Column has 299 outliers greater than upper bound (43.00) or lower than lower bound(-5.00). Cap them or remove them., Column has a high correlation with ['text_length']. Consider dropping one of them."


Number of All Scatter Plots = 3


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/berto/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/berto/nltk_data...


Could not draw wordcloud plot for author_id. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

All Plots done
Time to run AutoViz = 2 seconds 

 ###################### AUTO VISUALIZATION Completed ########################

AutoViz completed on 10K sample.


[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Zip Slip blocked: gazetteers/


In [5]:
# Cell 5 - Sentiment Analysis (GPU-Accelerated, 50K Sample)

# Filter inbound tweets
inbound_df = df[df['inbound'] == True].copy()
print(f"Total inbound tweets: {len(inbound_df):,}")

# Stratified sample of 50K for sentiment
sample_size = 50_000
sentiment_sample = inbound_df.sample(n=min(sample_size, len(inbound_df)), random_state=42).copy()
print(f"Sentiment sample: {len(sentiment_sample):,} tweets")

# Clean tweets
def clean_tweet(text):
    return re.sub(r'@\S+', '', str(text)).strip()

sentiment_sample['clean_text'] = sentiment_sample['text'].apply(clean_tweet)

# Remove empty texts after cleaning
sentiment_sample = sentiment_sample[sentiment_sample['clean_text'].str.len() > 0].copy()
print(f"After cleaning: {len(sentiment_sample):,} tweets")

# Load model
device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=device,
    batch_size=64
)

# Run with timing
print(f"\nRunning sentiment analysis (device={'GPU' if device == 0 else 'CPU'}, batch_size=64)...")
start = time.time()
results = sentiment_pipe(sentiment_sample['clean_text'].tolist(), truncation=True)
elapsed = time.time() - start

sentiment_sample['sentiment'] = [r['label'] for r in results]
sentiment_sample['confidence'] = [round(r['score'], 3) for r in results]

print(f"Completed in {elapsed:.1f}s ({len(sentiment_sample)/elapsed:.0f} tweets/sec)")
print(f"\n=== Sentiment Distribution ===")
dist = sentiment_sample['sentiment'].value_counts()
for label in ['negative', 'neutral', 'positive']:
    count = dist.get(label, 0)
    pct = count / len(sentiment_sample) * 100
    mean_conf = sentiment_sample[sentiment_sample['sentiment'] == label]['confidence'].mean()
    print(f"  {label:8s}: {count:,} ({pct:.1f}%) | mean confidence: {mean_conf:.3f}")

print(f"\nOverall mean confidence: {sentiment_sample['confidence'].mean():.3f}")

Total inbound tweets: 1,537,843
Sentiment sample: 50,000 tweets
After cleaning: 49,920 tweets


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running sentiment analysis (device=GPU, batch_size=64)...
Completed in 981.6s (51 tweets/sec)

=== Sentiment Distribution ===
  negative: 25,991 (52.1%) | mean confidence: 0.802
  neutral : 16,402 (32.9%) | mean confidence: 0.729
  positive: 7,527 (15.1%) | mean confidence: 0.821

Overall mean confidence: 0.781


In [6]:
# Cell 6 - Sentiment Visualization and Company Analysis
os.makedirs("outputs", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
colors = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}
counts = sentiment_sample['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
axes[0].set_title('Sentiment Distribution (50K Inbound Sample)')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 50, f"{val:,}", ha='center', fontweight='bold')

# Confidence by sentiment
for sentiment in ['negative', 'neutral', 'positive']:
    subset = sentiment_sample[sentiment_sample['sentiment'] == sentiment]['confidence']
    axes[1].hist(subset, bins=20, alpha=0.6, label=sentiment, color=colors[sentiment])
axes[1].set_title('Model Confidence by Sentiment')
axes[1].set_xlabel('Confidence Score')
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/challenge_50k_sentiment_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# Company-level sentiment
# Extract company from @mention in tweet text
sentiment_sample['company'] = sentiment_sample['text'].apply(
    lambda t: re.search(r'@(\w+)', str(t)).group(1) if re.search(r'@(\w+)', str(t)) else 'Unknown'
)

company_counts = sentiment_sample['company'].value_counts()
top_companies = company_counts[company_counts >= 50].index.tolist()
print(f"Companies with 50+ tweets: {len(top_companies)}")

top_df = sentiment_sample[sentiment_sample['company'].isin(top_companies)]
cross = pd.crosstab(top_df['company'], top_df['sentiment'], normalize='index') * 100
cross = cross.reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
cross = cross.sort_values('negative', ascending=True)

print(f"\n=== Sentiment by Company (% of tweets, 50+ tweets) ===")
print(cross.round(1).to_string())

# Stacked bar chart
fig, ax = plt.subplots(figsize=(12, max(6, len(cross) * 0.4)))
cross.plot(kind='barh', stacked=True, color=['#e74c3c', '#95a5a6', '#2ecc71'], ax=ax)
ax.set_xlabel('Percentage of Tweets')
ax.set_title('Sentiment by Company (50+ tweets)')
ax.legend(title='Sentiment')
plt.tight_layout()
plt.savefig("outputs/challenge_50k_sentiment_by_company.png", dpi=150, bbox_inches="tight")
plt.show()

Companies with 50+ tweets: 113

=== Sentiment by Company (% of tweets, 50+ tweets) ===
sentiment        negative  neutral  positive
company                                     
117086              3.5     82.5      14.0  
116928              7.9     74.6      17.5  
116875              8.2     70.5      21.3  
120533             10.0     81.4       8.6  
116316             11.3     73.6      15.1  
AirAsiaSupport     24.0     64.2      11.9  
NikeSupport        29.9     47.9      22.2  
AskPapaJohns       30.4     50.7      18.8  
azuresupport       30.6     67.7       1.6  
AskTarget          33.5     39.4      27.1  
nationalrailenq    33.8     45.3      20.9  
AzureSupport       34.2     44.1      21.7  
SouthwestAir       34.6     26.1      39.2  
AldiUK             34.8     22.3      43.0  
sizehelpteam       35.9     51.6      12.5  
Safaricom_Care     36.0     51.1      12.8  
McDonalds          36.3     25.8      37.9  
AskAmex            38.1     48.1      13.8  
GreggsOfficia

In [7]:
# Cell 7 - Scale Comparison: 93-Row Sample vs Full Dataset

print("=== W3 (93-row sample) vs Challenge (50K from 2.8M) ===\n")

w3_dist = {'negative': 55, 'neutral': 29, 'positive': 16}
challenge_dist = (sentiment_sample['sentiment'].value_counts(normalize=True) * 100).round(1)

comparison = pd.DataFrame({
    'W3 (93 rows, %)': [w3_dist.get(s, 0) for s in ['negative', 'neutral', 'positive']],
    'Challenge (50K, %)': [challenge_dist.get(s, 0) for s in ['negative', 'neutral', 'positive']]
}, index=['negative', 'neutral', 'positive'])

print(comparison.to_string())
print(f"\nW3 mean confidence: 0.782")
print(f"Challenge mean confidence: {sentiment_sample['confidence'].mean():.3f}")
print(f"\nW3 companies analyzed: 7 (3+ tweets)")
print(f"Challenge companies analyzed: {len(top_companies)} (50+ tweets)")

# Timing comparison
print(f"\nThroughput: {len(sentiment_sample)/elapsed:.0f} tweets/sec")
print(f"Estimated time for full inbound ({len(inbound_df):,} tweets): {len(inbound_df)/max(1, len(sentiment_sample)/elapsed)/60:.1f} min")

=== W3 (93-row sample) vs Challenge (50K from 2.8M) ===

          W3 (93 rows, %)  Challenge (50K, %)
negative        55                52.1       
neutral         29                32.9       
positive        16                15.1       

W3 mean confidence: 0.782
Challenge mean confidence: 0.781

W3 companies analyzed: 7 (3+ tweets)
Challenge companies analyzed: 113 (50+ tweets)

Throughput: 51 tweets/sec
Estimated time for full inbound (1,537,843 tweets): 504.0 min


## Scale Comparison and Reflections

### What Changes at Scale

Moving from 93 rows to 2.8 million fundamentally changes the analysis in three ways:

1. **Statistical significance.** Company-level sentiment patterns that were anecdotal at 93 rows (3-9 tweets per company) become statistically meaningful. The 50K sample identified 113 companies with 50+ tweets each, compared to 7 in the assignment and 49 in the 10K challenge.

2. **Sampling becomes mandatory.** AutoViz would crash on the full dataset; sentiment analysis on 1.5M inbound tweets without GPU batching would take over 8 hours. Every tool required explicit sampling decisions.

3. **Infrastructure matters.** GPU batching at 51 tweets/sec processed the 50K sample in 982 seconds (~16 min). Estimated time for the full inbound corpus: ~504 minutes. These are real engineering constraints, not academic ones.

### Sentiment Stability Across Scales

The distribution stabilized across three sample sizes:

| | W3 (93 rows) | 10K | 50K |
|---|---|---|---|
| Negative | 55% | 51.6% | 52.1% |
| Neutral | 29% | 33.4% | 32.9% |
| Positive | 16% | 15.0% | 15.1% |
| Confidence | 0.782 | 0.782 | 0.781 |
| Companies (50+) | 7 | 49 | 113 |

The 10K-to-50K shift was minimal (<0.5pp), confirming that the distribution had already converged at 10K. Mean confidence held steady at 0.78, suggesting the model's behavior is sample-size-independent for this dataset.

### Company-Level Insights

The 50K sample revealed a wide spectrum: negative sentiment ranged from 3.5% (numeric ID accounts, likely automated) to 87.3% (115858). Among named companies, DunkinDonuts (82.7% negative) and ComcastCares (73.8%) stood at the high end, while SouthwestAir (34.6%) and AldiUK (34.8%) had notably lower negativity. The numeric ID accounts with very low negativity are likely outbound/support accounts misclassified as inbound.

### Throughput

Throughput held constant at 51 tweets/sec across both 10K and 50K runs, confirming linear scaling on the Quadro T1000. At this rate, the full 1.5M inbound corpus would require ~504 minutes (~8.4 hours) of GPU time.

### Tool Performance at Scale

- **Gemini:** Scales well because it receives summary statistics, not raw data. The analysis quality depends on the summary, not the dataset size.
- **AutoViz:** Same limitation as W3; text-heavy data gives it little to work with regardless of scale.
- **Hugging Face:** GPU batching transforms feasibility. What would be impractical on CPU becomes routine with batch_size=64.

### Connection to the Series

The W3 assignment proved the tools work on a curated sample. The challenge notebooks tested whether the findings generalize: the negative sentiment skew holds, distributions converge by 10K, company patterns become statistically meaningful, and GPU acceleration makes the analysis practical at production scale.